# Programación funcional · Herramientas funcionales

Continuación de *Iteradores, generadores y decoradores* con las piezas que sostienen el estilo funcional en Python: ámbitos y *closures*, `*args` y `**kwargs`, aplicación parcial y currificación, composición de funciones, *callbacks*, más recetas de `itertools`, tuberías perezosas sobre generadores infinitos y corrutinas con `send`, `throw` y `close`. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/roadmap) y [PYnative](https://pynative.com/python-exercises-with-solutions/).

## 1. *Closures*, `global` y `nonlocal`

Crear una fábrica `crear_contador(inicio, paso)` que devuelva una función que, en cada llamada, avance y devuelva su propia cuenta, independiente de la de otros contadores. Ver qué ocurre si la función interna modifica la variable sin declararla `nonlocal`, inspeccionar las celdas que guarda la *closure* y compararlo con un contador de visitas en una variable `global`.

In [1]:
def crear_contador(inicio=0, paso=1):
    cuenta = inicio

    def siguiente():
        nonlocal cuenta
        cuenta += paso
        return cuenta

    return siguiente


turnos = crear_contador()
tickets = crear_contador(100, 10)
print("turnos: ", [turnos() for _ in range(3)])
print("tickets:", [tickets() for _ in range(3)])
print("turnos sigue en", turnos(), "sin verse afectado por tickets")
print("Celdas de la closure:", {nombre: celda.cell_contents
                                for nombre, celda in zip(tickets.__code__.co_freevars, tickets.__closure__)})
assert turnos() == 5 and tickets() == 140


def crear_contador_roto():
    cuenta = 0

    def siguiente():
        cuenta += 1  # la asignación la convierte en local de siguiente
        return cuenta

    return siguiente


try:
    crear_contador_roto()()
except UnboundLocalError as e:
    print("Sin nonlocal:", e)

turnos:  [1, 2, 3]
tickets: [110, 120, 130]
turnos sigue en 4 sin verse afectado por tickets
Celdas de la closure: {'cuenta': 130, 'paso': 10}
Sin nonlocal: cannot access local variable 'cuenta' where it is not associated with a value


In [2]:
visitas = 0


def registrar_visita():
    global visitas
    visitas += 1
    return visitas


for _ in range(4):
    registrar_visita()
print("Visitas (global):", visitas)
assert visitas == 4

Visitas (global): 4


## 2. La trampa de las lambdas en un bucle

Crear en un bucle tres funciones que calculen la tarifa de una guagua para 1, 2 y 3 zonas (`precio_base * zonas`). Comprobar que, creadas con `lambda` dentro de una comprensión, las tres usan la última zona (las *closures* leen la variable cuando se llaman, no cuando se crean) y arreglarlo de tres formas: valor por defecto, función fábrica y `functools.partial`.

In [3]:
from functools import partial
from operator import mul

precio_base = 1.40

tarifas = [lambda: precio_base * zonas for zonas in range(1, 4)]
print("Con el fallo:   ", [round(t(), 2) for t in tarifas])


def fabrica(zonas):
    return lambda: precio_base * zonas


arreglos = {
    "valor por defecto": [lambda zonas=zonas: precio_base * zonas for zonas in range(1, 4)],
    "función fábrica": [fabrica(zonas) for zonas in range(1, 4)],
    "partial": [partial(mul, precio_base, zonas) for zonas in range(1, 4)],
}
esperado = [1.40, 2.80, 4.20]
for nombre, funciones in arreglos.items():
    valores = [round(f(), 2) for f in funciones]
    print(f"{nombre:<16}", valores)
    assert valores == esperado
assert len({round(t(), 2) for t in tarifas}) == 1

Con el fallo:    [4.2, 4.2, 4.2]
valor por defecto [1.4, 2.8, 4.2]
función fábrica  [1.4, 2.8, 4.2]
partial          [1.4, 2.8, 4.2]


## 3. `*args`, `**kwargs` y desempaquetado

Escribir un registrador `registrar(nivel, plantilla, *valores, separador=" | ", **contexto)` que rellene la plantilla con los valores posicionales y añada al final el contexto como `clave=valor`. Después llamarlo desempaquetando una tupla y un diccionario con `*` y `**`, combinar dos diccionarios en la misma llamada y capturar el error de pasar dos veces el mismo argumento.

In [4]:
def registrar(nivel, plantilla, *valores, separador=" | ", **contexto):
    linea = f"[{nivel:<5}] {plantilla.format(*valores)}"
    if contexto:
        linea += separador + ", ".join(f"{clave}={valor!r}" for clave, valor in contexto.items())
    return linea


print(registrar("INFO", "Servidor arrancado"))
print(registrar("WARN", "Pedido {} retrasado {} minutos", 812, 25, isla="La Gomera", reparto="barco"))

mensaje = ("ERROR", "Pago de {:.2f} € rechazado por {}", 49.9, "saldo insuficiente")
usuario = {"usuario": "nayra", "canal": "app"}
peticion = {"ip": "10.0.0.7"}
print(registrar(*mensaje, separador=" · ", **usuario, **peticion))

try:
    registrar("INFO", "Hola", **{"nivel": "DEBUG"})
except TypeError as e:
    print("TypeError:", e)

assert registrar(*mensaje, **usuario) == registrar("ERROR", mensaje[1], 49.9, "saldo insuficiente",
                                                  usuario="nayra", canal="app")
assert registrar("INFO", "x", a=1).endswith(" | a=1")

[INFO ] Servidor arrancado
[WARN ] Pedido 812 retrasado 25 minutos | isla='La Gomera', reparto='barco'
[ERROR] Pago de 49.90 € rechazado por saldo insuficiente · usuario='nayra', canal='app', ip='10.0.0.7'
TypeError: registrar() got multiple values for argument 'nivel'


## 4. `functools.partial` y currificación

Obtener `cuadrado` y `cubo` a partir de `pow` fijando el exponente, y `binario` a partir de `int` fijando la base. Después currificar a mano una función de dos argumentos (suma) y otra de tres (volumen de un contenedor), de modo que reciban un argumento en cada llamada, y escribir `currificar(f)`, que haga lo mismo con cualquier función mirando su número de parámetros.

In [5]:
import inspect

cuadrado = partial(pow, exp=2)
cubo = partial(pow, exp=3)
binario = partial(int, base=2)
print([cuadrado(n) for n in range(1, 6)], [cubo(n) for n in range(1, 6)])
print([binario(b) for b in ("101", "1111", "10000000")])
assert [cuadrado(n) for n in range(10)] == [n**2 for n in range(10)]
assert binario("11111111") == 255


def suma(a):
    return lambda b: a + b


def volumen(largo):
    return lambda ancho: lambda alto: largo * ancho * alto


suma_iva = suma(7)
print("suma(3)(4) =", suma(3)(4), "| suma_iva(100) =", suma_iva(100))
contenedor_20 = volumen(6.06)(2.44)
print(f"Contenedor de 20 pies con 2,59 m de alto: {contenedor_20(2.59):.2f} m³")
assert suma(3)(4) == 7 and volumen(2)(3)(4) == 24


def currificar(f):
    n = len(inspect.signature(f).parameters)

    def recoger(*acumulados):
        if len(acumulados) == n:
            return f(*acumulados)
        return lambda x: recoger(*acumulados, x)

    return recoger()


def precio_final(base, descuento, igic):
    return round(base * (1 - descuento) * (1 + igic), 2)


por_pasos = currificar(precio_final)
rebajas = por_pasos(80)(0.25)
print("Precio con 25 % de descuento e IGIC del 7 %:", rebajas(0.07))
assert rebajas(0.07) == precio_final(80, 0.25, 0.07) == 64.2
assert currificar(lambda a, b, c, d: a * 1000 + b * 100 + c * 10 + d)(1)(2)(3)(4) == 1234

[1, 4, 9, 16, 25] [1, 8, 27, 64, 125]
[5, 15, 128]
suma(3)(4) = 7 | suma_iva(100) = 107
Contenedor de 20 pies con 2,59 m de alto: 38.30 m³
Precio con 25 % de descuento e IGIC del 7 %: 64.2


## 5. `componer` y `tuberia`

Escribir `componer(*funciones)`, que aplique las funciones de derecha a izquierda como en matemáticas ($f \circ g$), y `tuberia(*funciones)`, que las aplique de izquierda a derecha en el orden de lectura. Usarlas para convertir títulos de noticias en *slugs* para una URL: quitar espacios sobrantes, pasar a minúsculas, quitar tildes, quitar signos de puntuación y unir las palabras con guiones.

In [6]:
import string
import unicodedata
from functools import reduce


def componer(*funciones):
    return lambda x: reduce(lambda acumulado, f: f(acumulado), reversed(funciones), x)


def tuberia(*funciones):
    return lambda x: reduce(lambda acumulado, f: f(acumulado), funciones, x)


def sin_tildes(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")


def sin_puntuacion(texto):
    return "".join(c for c in texto if c not in string.punctuation + "¡¿«»…")


def con_guiones(texto):
    return "-".join(texto.split())


slug = tuberia(str.strip, str.lower, sin_tildes, sin_puntuacion, con_guiones)
slug_matematico = componer(con_guiones, sin_puntuacion, sin_tildes, str.lower, str.strip)

titulos = ["  ¡Llega la calima a Fuerteventura!  ", "¿Cuándo abre el Teide? Horarios 2026", "Récord de turistas en Lanzarote…"]
for titulo in titulos:
    print(f"{titulo!r:<42} -> {slug(titulo)}")
    assert slug(titulo) == slug_matematico(titulo) == con_guiones(sin_puntuacion(sin_tildes(titulo.strip().lower())))
assert slug("Ñandú en Güímar") == "nandu-en-guimar"
assert componer()(42) == tuberia()(42) == 42

'  ¡Llega la calima a Fuerteventura!  '    -> llega-la-calima-a-fuerteventura
'¿Cuándo abre el Teide? Horarios 2026'     -> cuando-abre-el-teide-horarios-2026
'Récord de turistas en Lanzarote…'         -> record-de-turistas-en-lanzarote


## 6. Transductor: `filter` y `map` en una sola pasada

Encadenar `filter` y `map` con listas intermedias recorre los datos dos veces. Un transductor transforma la función reductora en lugar de los datos: `filtrando(p)` y `mapeando(f)` reciben un reductor y devuelven otro. Con ellos y `componer`, quedarse con los precios (en céntimos) de más de 5 € y aplicarles el IGIC del 7 % recorriendo la lista una sola vez, y reutilizar la misma transformación para sumar en lugar de acumular en una lista.

In [7]:
from operator import add


def mapeando(f):
    return lambda reductor: lambda acumulado, x: reductor(acumulado, f(x))


def filtrando(predicado):
    return lambda reductor: lambda acumulado, x: reductor(acumulado, x) if predicado(x) else acumulado


def agregar(lista, x):
    lista.append(x)
    return lista


precios = [350, 1299, 499, 820, 75, 2450, 600, 510]
visitas = []


def caro(x):
    visitas.append(x)
    return x > 500


con_igic = mapeando(lambda x: round(x * 1.07))
transformacion = componer(filtrando(caro), con_igic)

resultado = reduce(transformacion(agregar), precios, [])
print("Precios con IGIC:", resultado)
print("Elementos visitados por el filtro:", len(visitas), "de", len(precios))
total = reduce(transformacion(add), precios, 0)
print("Total con IGIC:", total, "céntimos")
assert resultado == list(map(lambda x: round(x * 1.07), filter(lambda x: x > 500, precios)))
assert total == sum(resultado) and len(visitas) == 2 * len(precios)

Precios con IGIC: [1390, 877, 2622, 642, 546]
Elementos visitados por el filtro: 8 de 8
Total con IGIC: 6077 céntimos


## 7. `Quizas`: encadenar sin comprobar `None`

Crear una clase `Quizas` que envuelva un valor que puede ser `None`. Su método `bind(f)` aplica `f` solo si hay valor (y envuelve el resultado), de modo que, en cuanto un paso devuelve `None`, el resto de la cadena se salta. Usarla para obtener el código postal (como entero) de la dirección de un cliente en un diccionario anidado donde puede faltar cualquier nivel, y compararla con la versión de `if` anidados.

In [8]:
class Quizas:
    __slots__ = ("valor",)

    def __init__(self, valor):
        self.valor = valor

    def bind(self, f):
        if self.valor is None:
            return self
        resultado = f(self.valor)
        return resultado if isinstance(resultado, Quizas) else Quizas(resultado)

    def o_si_no(self, por_defecto):
        return por_defecto if self.valor is None else self.valor

    def __repr__(self):
        return f"Quizas({self.valor!r})"


clientes = {
    "ana": {"direccion": {"isla": "Tenerife", "cp": "38001"}},
    "luis": {"direccion": None},
    "eva": {"telefono": "922000000"},
    "iván": {"direccion": {"isla": "El Hierro"}},
}
pasos = []


def paso(nombre, f):
    def envuelta(x):
        pasos.append(nombre)
        return f(x)
    return envuelta


def codigo_postal(nombre):
    return (Quizas(clientes.get(nombre))
            .bind(paso("direccion", lambda c: c.get("direccion")))
            .bind(paso("cp", lambda d: d.get("cp")))
            .bind(paso("int", int)))


def codigo_postal_con_ifs(nombre):
    cliente = clientes.get(nombre)
    if cliente is not None:
        direccion = cliente.get("direccion")
        if direccion is not None:
            cp = direccion.get("cp")
            if cp is not None:
                return int(cp)
    return None


for nombre in [*clientes, "nadie"]:
    pasos.clear()
    q = codigo_postal(nombre)
    print(f"{nombre:<6} {q!r:<16} {q.o_si_no('desconocido')!s:<12} pasos ejecutados: {pasos}")
    assert q.valor == codigo_postal_con_ifs(nombre)

ana    Quizas(38001)    38001        pasos ejecutados: ['direccion', 'cp', 'int']
luis   Quizas(None)     desconocido  pasos ejecutados: ['direccion']
eva    Quizas(None)     desconocido  pasos ejecutados: ['direccion']
iván   Quizas(None)     desconocido  pasos ejecutados: ['direccion', 'cp']
nadie  Quizas(None)     desconocido  pasos ejecutados: []


## 8. *Callbacks*: el restaurante avisa

Simular un restaurante que prepara pedidos. `procesar_pedido` recibe el pedido, un generador aleatorio y tres funciones opcionales (`al_confirmar`, `al_estar_listo`, `al_entregar`) a las que llama con el pedido y el minuto simulado de cada etapa. Pasarle un *callback* que avise al cliente por pantalla y otro que guarde un historial, y comprobar que cada pedido pasa por las tres etapas en orden y con tiempos crecientes.

In [9]:
import random


def procesar_pedido(pedido, rng, al_confirmar=None, al_estar_listo=None, al_entregar=None):
    minuto = pedido["llegada"]
    etapas = [(al_confirmar, 1, 3), (al_estar_listo, 8, 25), (al_entregar, 10, 30)]
    for callback, minimo, maximo in etapas:
        minuto += rng.randint(minimo, maximo)
        if callback is not None:
            callback(pedido, minuto)
    return minuto


def crear_aviso(texto):
    def avisar(pedido, minuto):
        print(f"  [min {minuto:>3}] {pedido['cliente']:<7} {texto} ({pedido['plato']})")
    return avisar


historial = []


def guardar(etapa):
    return lambda pedido, minuto: historial.append((pedido["id"], etapa, minuto))


def todos(*callbacks):
    def llamar(*args):
        for callback in callbacks:
            callback(*args)
    return llamar


pedidos = [
    {"id": 1, "cliente": "Ayoze", "plato": "papas arrugadas con mojo", "llegada": 0},
    {"id": 2, "cliente": "Nira", "plato": "ropa vieja", "llegada": 5},
    {"id": 3, "cliente": "Echedey", "plato": "sancocho", "llegada": 12},
]
rng = random.Random(2026)
for pedido in pedidos:
    print(f"Pedido {pedido['id']}:")
    procesar_pedido(pedido, rng,
                    al_confirmar=todos(crear_aviso("pedido confirmado"), guardar("confirmado")),
                    al_estar_listo=todos(crear_aviso("listo en cocina"), guardar("listo")),
                    al_entregar=todos(crear_aviso("entregado, ¡buen provecho!"), guardar("entregado")))

for id_pedido in (1, 2, 3):
    eventos = [(etapa, minuto) for i, etapa, minuto in historial if i == id_pedido]
    assert [e for e, _ in eventos] == ["confirmado", "listo", "entregado"]
    assert all(a < b for (_, a), (_, b) in zip(eventos, eventos[1:]))
sin_avisos = procesar_pedido(pedidos[0], random.Random(1))
assert sin_avisos == procesar_pedido(pedidos[0], random.Random(1))
print("Historial:", historial[:3], "...")

Pedido 1:
  [min   1] Ayoze   pedido confirmado (papas arrugadas con mojo)
  [min  19] Ayoze   listo en cocina (papas arrugadas con mojo)
  [min  45] Ayoze   entregado, ¡buen provecho! (papas arrugadas con mojo)
Pedido 2:
  [min   8] Nira    pedido confirmado (ropa vieja)
  [min  19] Nira    listo en cocina (ropa vieja)
  [min  36] Nira    entregado, ¡buen provecho! (ropa vieja)
Pedido 3:
  [min  15] Echedey pedido confirmado (sancocho)
  [min  40] Echedey listo en cocina (sancocho)
  [min  63] Echedey entregado, ¡buen provecho! (sancocho)
Historial: [(1, 'confirmado', 1), (1, 'listo', 19), (1, 'entregado', 45)] ...


## 9. `cycle` y `repeat`

Con `itertools.cycle`, repartir las guardias de cuatro farmacias de un pueblo durante los 10 primeros días de noviembre, y el turno de palabra entre tres personas en una reunión. Con `itertools.repeat` y `map`/`zip`, elevar al cuadrado una lista con `pow`, etiquetar varios municipios con la misma isla y crear una fila de asientos libres.

In [10]:
from datetime import date, timedelta
from itertools import cycle, islice, repeat

farmacias = ["Farmacia del Puerto", "Farmacia La Plaza", "Farmacia San Juan", "Farmacia El Charco"]
DIAS_SEMANA = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
dias = (date(2026, 11, 1) + timedelta(days=i) for i in range(10))
guardias = list(zip(dias, cycle(farmacias)))
for dia, farmacia in guardias:
    print(f"{DIAS_SEMANA[dia.weekday()]} {dia:%d/%m}  {farmacia}")
assert [f for _, f in guardias] == [farmacias[i % 4] for i in range(10)]

turno = cycle(["Ana", "Beatriz", "Carlos"])
print("Turnos de palabra:", list(islice(turno, 7)))

dom 01/11  Farmacia del Puerto
lun 02/11  Farmacia La Plaza
mar 03/11  Farmacia San Juan
mié 04/11  Farmacia El Charco
jue 05/11  Farmacia del Puerto
vie 06/11  Farmacia La Plaza
sáb 07/11  Farmacia San Juan
dom 08/11  Farmacia El Charco
lun 09/11  Farmacia del Puerto
mar 10/11  Farmacia La Plaza
Turnos de palabra: ['Ana', 'Beatriz', 'Carlos', 'Ana', 'Beatriz', 'Carlos', 'Ana']


In [11]:
numeros = [3, 7, 12, 25]
cuadrados = list(map(pow, numeros, repeat(2)))
municipios = list(zip(["Agaete", "Gáldar", "Teror"], repeat("Gran Canaria")))
fila = list(repeat("libre", 6))
print(cuadrados, municipios, fila, sep="\n")
assert cuadrados == [n**2 for n in numeros] and fila == ["libre"] * 6
assert all(isla == "Gran Canaria" for _, isla in municipios)

[9, 49, 144, 625]
[('Agaete', 'Gran Canaria'), ('Gáldar', 'Gran Canaria'), ('Teror', 'Gran Canaria')]
['libre', 'libre', 'libre', 'libre', 'libre', 'libre']


## 10. `dropwhile`, `takewhile` y `compress`

Un sensor de temperatura de un invernadero envía `0.0` mientras se calibra y después lecturas reales. Con `dropwhile`, descartar la calibración, y con `takewhile`, quedarse con las lecturas hasta la primera que supere la alarma de 35 °C. Ver en qué se diferencian de `filter`. Por último, con `compress` y una máscara de existencias, listar los productos que quedan en el almacén.

In [12]:
from itertools import compress, dropwhile, takewhile

lecturas = [0.0, 0.0, 0.0, 24.5, 26.1, 0.0, 29.8, 33.2, 36.4, 31.0, 28.7]
utiles = list(dropwhile(lambda t: t == 0.0, lecturas))
hasta_alarma = list(takewhile(lambda t: t <= 35, utiles))
print("Sin calibración:   ", utiles)
print("Hasta la alarma:   ", hasta_alarma)
print("filter (t != 0):   ", list(filter(lambda t: t != 0.0, lecturas)))
assert utiles == lecturas[3:] and 0.0 in utiles
assert hasta_alarma == [24.5, 26.1, 0.0, 29.8, 33.2]

productos = ["gofio", "plátanos", "queso majorero", "mojo picón", "bienmesabe"]
existencias = [12, 0, 4, 0, 7]
disponibles = list(compress(productos, (n > 0 for n in existencias)))
print("Con existencias:   ", disponibles)
assert disponibles == [p for p, n in zip(productos, existencias) if n > 0]

Sin calibración:    [24.5, 26.1, 0.0, 29.8, 33.2, 36.4, 31.0, 28.7]
Hasta la alarma:    [24.5, 26.1, 0.0, 29.8, 33.2]
filter (t != 0):    [24.5, 26.1, 29.8, 33.2, 36.4, 31.0, 28.7]
Con existencias:    ['gofio', 'queso majorero', 'bienmesabe']


## 11. `combinations_with_replacement` y `accumulate`

Una heladería tiene cinco sabores y vende copas de dos bolas en las que se puede repetir sabor y no importa el orden. Listar todas las copas con `combinations_with_replacement` y comprobar que hay $\binom{n+1}{2}$. Después usar `accumulate` con funciones distintas de la suma: el máximo para el récord de temperatura hasta cada día y el producto para el capital de un depósito con un interés distinto cada año.

In [13]:
from itertools import accumulate, combinations_with_replacement, product
from math import comb, isclose, prod

sabores = ["gofio", "plátano", "vainilla", "chocolate", "fresa"]
copas = list(combinations_with_replacement(sabores, 2))
print(len(copas), "copas:", ", ".join(" + ".join(c) for c in copas[:6]), "...")
assert len(copas) == comb(len(sabores) + 1, 2) == 15
assert {tuple(sorted(c)) for c in copas} == {tuple(sorted(c)) for c in product(sabores, repeat=2)}

maximas = [24, 26, 25, 29, 28, 31, 27, 33]
records = list(accumulate(maximas, max))
print("Máximas:", maximas)
print("Récords:", records)
assert records == [max(maximas[:i + 1]) for i in range(len(maximas))]

intereses = [0.02, 0.025, 0.03, 0.015]
capital = list(accumulate((1 + r for r in intereses), lambda c, f: round(c * f, 2), initial=1000))
print("Capital año a año:", capital)
assert isclose(capital[-1], 1000 * prod(1 + r for r in intereses), abs_tol=0.05)

15 copas: gofio + gofio, gofio + plátano, gofio + vainilla, gofio + chocolate, gofio + fresa, plátano + plátano ...
Máximas: [24, 26, 25, 29, 28, 31, 27, 33]
Récords: [24, 26, 26, 29, 29, 31, 31, 33]
Capital año a año: [1000, 1020.0, 1045.5, 1076.87, 1093.02]


## 12. Tubería perezosa sobre un generador infinito

A partir de un generador infinito de naturales, construir una tubería perezosa: elevar al cuadrado (`map`), quedarse con los que acaban en 1 o en 9 (`filter`), tomar los 8 primeros (`islice`) y sumarlos con `functools.reduce`. Envolver el generador para contar cuántos números llega a producir y comprobar que solo genera los que necesita.

In [14]:
producidos = 0


def naturales():
    global producidos
    n = 1
    while True:
        producidos += 1
        yield n
        n += 1


cuadrados = map(lambda n: n * n, naturales())
acaban_en_1_o_9 = filter(lambda c: c % 10 in (1, 9), cuadrados)
primeros = islice(acaban_en_1_o_9, 8)
print("Antes de consumir se han producido", producidos, "números")
seleccion = list(primeros)
total = reduce(add, seleccion)
print("Cuadrados elegidos:", seleccion, "| suma:", total)
print("Números producidos:", producidos)
assert seleccion == [n * n for n in range(1, 21) if (n * n) % 10 in (1, 9)][:8]
assert total == sum(seleccion) and producidos == 19

Antes de consumir se han producido 0 números
Cuadrados elegidos: [1, 9, 49, 81, 121, 169, 289, 361] | suma: 1080
Números producidos: 19


## 13. Lotes, media móvil y ventanas deslizantes

Escribir tres generadores que funcionen con cualquier iterable, también infinito: `lotes(iterable, n)`, que agrupe en tuplas de `n` elementos (la última puede ser más corta); `media_movil(iterable, k)`, que produzca la media de los últimos `k` valores manteniendo una suma acumulada; y `ventanas(iterable, k)`, que produzca cada tramo de `k` elementos consecutivos. Comprobarlos con `itertools.batched` y con cortes de listas.

In [15]:
from collections import deque
from itertools import batched, count


def lotes(iterable, n):
    iterador = iter(iterable)
    while lote := tuple(islice(iterador, n)):
        yield lote


def media_movil(iterable, k):
    ventana = deque()
    suma = 0
    for x in iterable:
        ventana.append(x)
        suma += x
        if len(ventana) > k:
            suma -= ventana.popleft()
        if len(ventana) == k:
            yield suma / k


def ventanas(iterable, k):
    ventana = deque(maxlen=k)
    for x in iterable:
        ventana.append(x)
        if len(ventana) == k:
            yield tuple(ventana)


pasajeros = [120, 95, 143, 80, 160, 175, 90, 110, 130, 150]
print("Lotes de 4:  ", list(lotes(pasajeros, 4)))
print("Media de 3:  ", [round(m, 1) for m in media_movil(pasajeros, 3)])
print("Ventanas de 3:", list(ventanas(pasajeros, 3))[:3], "...")
print("Sobre un infinito:", list(islice(lotes(count(), 3), 3)))

random.seed(5)
for _ in range(100):
    datos = [random.randint(0, 50) for _ in range(random.randint(0, 20))]
    k = random.randint(1, 6)
    assert list(lotes(datos, k)) == list(batched(datos, k))
    assert list(ventanas(datos, k)) == [tuple(datos[i:i + k]) for i in range(len(datos) - k + 1)]
    assert all(isclose(m, sum(v) / k) for m, v in zip(media_movil(datos, k), ventanas(datos, k), strict=True))

Lotes de 4:   [(120, 95, 143, 80), (160, 175, 90, 110), (130, 150)]
Media de 3:   [119.3, 106.0, 127.7, 138.3, 141.7, 125.0, 110.0, 130.0]
Ventanas de 3: [(120, 95, 143), (95, 143, 80), (143, 80, 160)] ...
Sobre un infinito: [(0, 1, 2), (3, 4, 5), (6, 7, 8)]


## 14. Criba de Eratóstenes perezosa

Escribir un generador infinito de primos basado en la criba de Eratóstenes que no necesite conocer el límite de antemano: guarda en un diccionario el siguiente múltiplo pendiente de cada primo encontrado. Si un número no está en el diccionario es primo y se apunta su cuadrado; si está, se mueven sus primos a su siguiente múltiplo. Obtener los 15 primeros primos y el que ocupa el puesto 10 000, y comprobarlo contra una criba clásica.

In [16]:
def primos():
    pendientes = {}
    for n in count(2):
        if n not in pendientes:
            yield n
            pendientes[n * n] = [n]
        else:
            for p in pendientes.pop(n):
                pendientes.setdefault(n + p, []).append(p)


def criba_clasica(limite):
    es_primo = [True] * (limite + 1)
    es_primo[0:2] = [False, False]
    for i in range(2, int(limite**0.5) + 1):
        if es_primo[i]:
            es_primo[i * i::i] = [False] * len(range(i * i, limite + 1, i))
    return [i for i, p in enumerate(es_primo) if p]


print("Primeros 15:", list(islice(primos(), 15)))
diezmil = next(islice(primos(), 9999, None))
print("Primo número 10 000:", diezmil)
assert list(islice(primos(), 10_000)) == criba_clasica(diezmil)
assert list(takewhile(lambda p: p < 100, primos())) == criba_clasica(99)

Primeros 15: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Primo número 10 000: 104729


## 15. Corrutinas con generadores: `send`, `throw` y `close`

Un generador también puede recibir valores: `x = yield resultado` devuelve lo que se le pase con `send`. Escribir un decorador `cebada` que lo avance hasta el primer `yield` y una corrutina `media_acumulada` que devuelva la media de todo lo recibido. Después escribir `registro_sensor`, que guarde lecturas, trate con `throw` las averías como lecturas perdidas sin detenerse y, al cerrarla con `close`, cierre el registro en un bloque `try/finally`.

In [17]:
from functools import wraps


def cebada(funcion_generadora):
    @wraps(funcion_generadora)
    def crear(*args, **kwargs):
        corrutina = funcion_generadora(*args, **kwargs)
        next(corrutina)
        return corrutina
    return crear


@cebada
def media_acumulada():
    total = cantidad = 0
    media = None
    while True:
        x = yield media
        total += x
        cantidad += 1
        media = total / cantidad


notas = [7, 9, 4, 8, 10]
media = media_acumulada()
medias = [media.send(n) for n in notas]
print("Medias:", [round(m, 2) for m in medias])
assert medias == [sum(notas[:i + 1]) / (i + 1) for i in range(len(notas))]

Medias: [7.0, 8.0, 6.67, 7.0, 7.6]


In [18]:
@cebada
def registro_sensor(registro):
    registro.append("abierto")
    perdidas = 0
    try:
        while True:
            try:
                lectura = yield perdidas
                registro.append(lectura)
            except ConnectionError as e:
                perdidas += 1
                registro.append(f"perdida ({e})")
    finally:
        registro.append("cerrado")


registro = []
sensor = registro_sensor(registro)
sensor.send(21.5)
sensor.send(22.0)
print("Lecturas perdidas tras throw:", sensor.throw(ConnectionError("cable suelto")))
sensor.send(22.4)
sensor.close()
print("Registro:", registro)
try:
    sensor.send(23.0)
except StopIteration:
    print("Tras close, send lanza StopIteration")
assert registro == ["abierto", 21.5, 22.0, "perdida (cable suelto)", 22.4, "cerrado"]

Lecturas perdidas tras throw: 1
Registro: ['abierto', 21.5, 22.0, 'perdida (cable suelto)', 22.4, 'cerrado']
Tras close, send lanza StopIteration


## 16. `zip` propio y turnos alternos

Reimplementar `zip` con `iter` y `next`, incluido el modo `estricto` que lanza `ValueError` si los iterables no tienen la misma longitud. Después escribir `turnos_alternos(*iterables)`, que tome un elemento de cada iterable por turnos y, cuando uno se agote, siga con los demás hasta terminarlos todos (como repartir las preguntas de un concurso entre equipos con distinto número de preguntas). Comprobar ambos contra `zip` y `zip_longest`.

In [19]:
from itertools import zip_longest


def mi_zip(*iterables, estricto=False):
    if not iterables:
        return
    iteradores = [iter(it) for it in iterables]
    while True:
        fila = []
        for i, iterador in enumerate(iteradores):
            try:
                fila.append(next(iterador))
            except StopIteration:
                # Sin capturarla, PEP 479 la convertiría en RuntimeError dentro del generador
                if estricto:
                    centinela = object()
                    if i > 0 or any(next(otro, centinela) is not centinela for otro in iteradores[1:]):
                        raise ValueError(f"el iterable {i + 1} no tiene la misma longitud que los demás") from None
                return
        yield tuple(fila)


def turnos_alternos(*iterables):
    pendientes = deque(iter(it) for it in iterables)
    while pendientes:
        iterador = pendientes.popleft()
        try:
            x = next(iterador)
        except StopIteration:
            continue
        yield x
        pendientes.append(iterador)


print(list(mi_zip("ABC", [1, 2, 3, 4], count(10))))
try:
    list(mi_zip([1, 2], [3], estricto=True))
except ValueError as e:
    print("Estricto:", e)
equipos = (["A1", "A2", "A3"], ["B1"], ["C1", "C2"])
print("Concurso:", list(turnos_alternos(*equipos)))

random.seed(11)
for _ in range(200):
    listas = [list(range(random.randint(0, 5))) for _ in range(random.randint(0, 4))]
    assert list(mi_zip(*listas)) == list(zip(*listas))
    centinela = object()
    assert list(turnos_alternos(*listas)) == [x for fila in zip_longest(*listas, fillvalue=centinela)
                                             for x in fila if x is not centinela]
    try:
        esperado = list(zip(*listas, strict=True))
    except ValueError:
        esperado = ValueError
    try:
        obtenido = list(mi_zip(*listas, estricto=True))
    except ValueError:
        obtenido = ValueError
    assert obtenido == esperado

[('A', 1, 10), ('B', 2, 11), ('C', 3, 12)]
Estricto: el iterable 2 no tiene la misma longitud que los demás
Concurso: ['A1', 'B1', 'C1', 'A2', 'C2', 'A3']


## 17. Iterador con `mirar` y semáforo cíclico

Crear `IteradorConMirada`, un iterador que envuelve a otro y añade `mirar()`, que devuelve el siguiente elemento sin consumirlo (o un valor por defecto si no quedan). Usarlo para dividir una expresión en números de varias cifras y operadores leyendo carácter a carácter. Después escribir un generador infinito `semaforo` que produzca el color de cada segundo a partir de las duraciones de verde, ámbar y rojo, y mostrar los primeros 20 segundos.

In [20]:
import re


class IteradorConMirada:
    _NADA = object()

    def __init__(self, iterable):
        self._iterador = iter(iterable)
        self._guardado = self._NADA

    def __iter__(self):
        return self

    def __next__(self):
        if self._guardado is not self._NADA:
            x, self._guardado = self._guardado, self._NADA
            return x
        return next(self._iterador)

    def mirar(self, por_defecto=None):
        if self._guardado is self._NADA:
            self._guardado = next(self._iterador, self._NADA)
        return por_defecto if self._guardado is self._NADA else self._guardado


def tokenizar(expresion):
    caracteres = IteradorConMirada(expresion.replace(" ", ""))
    for c in caracteres:
        if c.isdigit():
            while caracteres.mirar("").isdigit():
                c += next(caracteres)
        yield c


for expresion in ["12+345*6", "(7 - 20) / 400"]:
    print(f"{expresion!r:<18} -> {list(tokenizar(expresion))}")
    assert list(tokenizar(expresion)) == re.findall(r"\d+|\S", expresion)

it = IteradorConMirada([1, 2])
assert (it.mirar(), it.mirar(), next(it), next(it), it.mirar("fin")) == (1, 1, 1, 2, "fin")

'12+345*6'         -> ['12', '+', '345', '*', '6']
'(7 - 20) / 400'   -> ['(', '7', '-', '20', ')', '/', '400']


In [21]:
def semaforo(duraciones):
    for color, segundos in cycle(duraciones.items()):
        yield from repeat(color, segundos)


duraciones = {"verde": 5, "ámbar": 2, "rojo": 4}
primeros = list(islice(semaforo(duraciones), 20))
print(" ".join(c[0].upper() for c in primeros))
ciclo = sum(duraciones.values())
assert all(primeros[i] == primeros[i + ciclo] for i in range(20 - ciclo))
assert primeros[:ciclo].count("ámbar") == 2 and primeros[5:7] == ["ámbar", "ámbar"]

V V V V V Á Á R R R R V V V V V Á Á R R
